# 🎬 EDA: La industria del cine (1980-2019)

**Dataset principal:** [TMDB + IMDB Merged Movies Dataset](https://www.kaggle.com/datasets/ggtejas/tmdb-imdb-merged-movies-dataset) · Kaggle  
**Dataset premios:** Oscar Awards Dataset (Academy Awards 1927-2023)

Este análisis explora qué factores influyen en el éxito comercial y el reconocimiento crítico de las películas:
- ¿Más presupuesto garantiza más recaudación?
- ¿Qué géneros son más rentables?
- ¿Los premios y la valoración del público van de la mano?
- ¿Qué tienen en común las películas más galardonadas?

---

## 1. Preparación del dataset

### 1.1 Carga y limpieza inicial

El dataset original se pre-procesó en **Power Query** para unificar formatos y eliminar columnas irrelevantes, exportando el resultado como `tmdb_imdb_limpio.csv`.

Primeros filtros aplicados:
- Eliminar filas sin género (variable central del análisis)
- Eliminar filas con `budget` o `revenue` igual a 0 (datos ausentes enmascarados como ceros)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy import stats

In [ ]:
df_movies = pd.read_csv("data/tmdb_imdb_limpio.csv", sep=";")
df_movies.info()

In [ ]:
# Eliminar filas sin género
df = df_movies.dropna(subset="genre")

# Budget y revenue = 0 son datos ausentes, no valores reales
df = df[df["budget"] != 0]
df = df[df["revenue"] != 0]

print(f"Registros tras limpieza inicial: {len(df):,}")

### 1.2 Acotación temporal

Se analiza la distribución temporal para decidir el rango de años a estudiar.

In [ ]:
df['year'] = pd.to_datetime(df['release_date']).dt.year
df['year'].value_counts().sort_index().plot(kind='line', figsize=(12, 4), title='Películas por año')
plt.xlabel('Año')
plt.ylabel('Número de películas')
plt.tight_layout()
plt.show()

La distribución muestra muy pocos registros antes de 1980 y una caída brusca a partir de 2019 (inicio de la pandemia). Se acota el estudio a **1980-2019** para trabajar con datos representativos.

In [ ]:
df_acotado = df[(df['year'] >= 1980) & (df['year'] <= 2019)]
print(f"Registros en el rango 1980-2019: {len(df_acotado):,}")

### 1.3 Incorporación de datos de los Óscar

Se integra el histórico de nominaciones y premios de la Academia.

**Decisiones de limpieza:**
- Los registros sin `FilmId` corresponden a premios técnicos y honoríficos → se eliminan
- Se acota al mismo rango temporal (hasta 2020; los premios se entregan con un año de retraso)
- La columna `Winner` llega con `NaN` para los no ganadores → se reemplaza por `False`

In [ ]:
oscars = pd.read_csv("data/Oscar_Awards.csv", sep="\t")

# Normalizar año (algunos están en formato "1927/28")
oscars['Year'] = oscars['Year'].str[:4].astype(int)

# Filtrar rango temporal y eliminar registros sin FilmId
oscars_acotado = oscars[(oscars['Year'] >= 1980) & (oscars['Year'] <= 2020)]
oscars_acotado = oscars_acotado[oscars_acotado['FilmId'].notnull()]
oscars_acotado = oscars_acotado[['Year', 'FilmId', 'Category', 'Winner']]

# NaN en Winner = no ganador
oscars_acotado = oscars_acotado.fillna(False)
oscars_acotado['Winner'] = oscars_acotado['Winner'].astype(int)

print(f"Registros de premios: {len(oscars_acotado):,}")

In [ ]:
# Agregar por película: total nominaciones y premios
awards_and_nominees = (
    oscars_acotado
    .groupby('FilmId')['Winner']
    .agg(['count', 'sum'])
    .rename(columns={'count': 'nominees', 'sum': 'awards'})
    .reset_index()
    .rename(columns={'FilmId': 'imdb_id'})
)

awards_and_nominees.head()

### 1.4 Merge y construcción del dataset final

Se hace un `left join` para conservar todas las películas, añadiendo los datos de premios donde existan.

> **Nota:** Solo 1.095 de las películas del dataset tienen registros en el dataset de los Óscar. El resto no aparece porque les faltaban datos de presupuesto, recaudación o género y fueron filtradas previamente.

In [ ]:
df_pelisfinal = df_acotado.merge(awards_and_nominees, how='left', on='imdb_id')

# Rellenar con 0 las películas sin nominaciones ni premios
df_pelisfinal['awards'] = df_pelisfinal['awards'].fillna(0)
df_pelisfinal['nominees'] = df_pelisfinal['nominees'].fillna(0)

print(f"Dataset final: {len(df_pelisfinal):,} películas")
df_pelisfinal.info()

### 1.5 Selección de columnas y métricas derivadas

Columnas descartadas por baja calidad o redundancia:
- `popularity`: mide búsquedas e impacto web, no valoración
- `vote_average`: distribución anómala, probablemente corrompida durante el merge
- `vote_count`: sin uso en el análisis

Se conserva `averageRating` (IMDB) como medida de valoración del público y se añade el **ROI** como métrica de rentabilidad.

In [ ]:
pelis_prep = df_pelisfinal[
    list(df_pelisfinal.columns[:15]) + ['averageRating'] + list(df_pelisfinal.columns[-3:])
]

pelis_prep = pelis_prep.drop(columns=['popularity', 'vote_average', 'vote_count'])

# ROI: recaudación por cada dólar invertido
pelis_prep['ROI'] = pelis_prep['revenue'] / pelis_prep['budget']

pelis_prep.info()

### 1.6 Tratamiento de valores extremos en budget y revenue

Los boxplots revelan valores sospechosamente bajos. Se contrastan manualmente en [Box Office Mojo](https://www.boxofficemojo.com).

**Conclusiones:**
- `budget < 10.000 $`: la mediana de estos casos es 9.150 $, claramente errónea para una producción cinematográfica → se eliminan
- `revenue < 5.000 $`: algunos son correctos pero solo recogen recaudación doméstica parcial, no comparables con el resto → se eliminan

In [ ]:
cols = ["revenue", "runtime", "budget", "averageRating", "ROI"]
fig, axes = plt.subplots(1, len(cols), figsize=(20, 5))
for i, col in enumerate(cols):
    sns.boxplot(data=pelis_prep[col], ax=axes[i])
    axes[i].set_title(col)
plt.suptitle("Distribución de variables numéricas antes del filtrado de outliers", y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
pelis_prep = pelis_prep[pelis_prep['budget'] >= 10_000]
pelis_prep = pelis_prep[pelis_prep['revenue'] >= 5_000]

print(f"Dataset limpio: {len(pelis_prep):,} películas")

### 1.7 Guardado del dataset procesado

Se exporta el dataset limpio para retomar el análisis sin reejecutar el preprocesado.

In [ ]:
pelis_prep.to_csv("data/pelis_ppal.csv", sep=";", index=False)
print("Guardado en data/pelis_ppal.csv")

---
## 2. Análisis exploratorio

A partir de aquí se carga el dataset procesado. Esta sección es independiente del preprocesado anterior.

In [ ]:
pelis = pd.read_csv("data/pelis_ppal.csv", sep=";")

# Columnas en millones para facilitar la lectura de gráficos
pelis['budget_M'] = pelis['budget'] / 1e6
pelis['revenue_M'] = pelis['revenue'] / 1e6

# Variable auxiliar: si la película tiene al menos una nominación
pelis['nominadas'] = pelis['nominees'] > 0

print(f"Dataset cargado: {len(pelis):,} películas | {pelis['year'].min()}-{pelis['year'].max()}")

### 2.1 Correlaciones entre variables numéricas

In [ ]:
sns.heatmap(
    pelis.select_dtypes(include='number').corr().round(2),
    annot=True, fmt=".2f", cmap="Blues",
    linewidths=0.5, figsize=(10, 8)
)
plt.title('Correlación entre variables numéricas')
plt.tight_layout()
plt.show()

**Principales correlaciones:**
- `budget` ↔ `revenue` (0.76): más presupuesto tiende a generar más recaudación
- `nominees` ↔ `awards` (0.77): más nominaciones aumenta la probabilidad de ganar
- `averageRating` muestra correlación moderada con nominaciones pero baja con recaudación

### 2.2 Presupuesto vs. recaudación

¿Las películas con mayor presupuesto recaudan más? ¿Las nominadas se concentran en algún rango?

In [ ]:
fig = px.scatter(
    pelis,
    x='budget_M', y='revenue_M',
    color='nominadas',
    hover_name='title',
    color_discrete_map={True: '#D4AF37', False: '#4A7FB5'},
    labels={
        'budget_M': 'Presupuesto (mill $)',
        'revenue_M': 'Recaudación (mill $)',
        'nominadas': 'Nominada al Óscar'
    },
    title='Presupuesto vs. Recaudación',
    opacity=0.6
)
fig.update_layout(template='plotly_dark')
fig.show()

Hay una correlación clara entre presupuesto y recaudación. Las nominadas tienden a concentrarse en el rango medio-alto, aunque con excepciones relevantes en ambos extremos.

### 2.3 Presupuesto vs. ROI

Un presupuesto alto no garantiza rentabilidad proporcional. Los picos más altos de ROI se dan en películas de bajo coste.

In [ ]:
fig = px.scatter(
    pelis,
    x='budget_M', y='ROI',
    color='nominadas',
    hover_name='title',
    color_discrete_map={True: '#D4AF37', False: '#4A7FB5'},
    labels={
        'budget_M': 'Presupuesto (mill $)',
        'ROI': 'ROI',
        'nominadas': 'Nominada al Óscar'
    },
    title='Presupuesto vs. Rentabilidad (ROI)',
    opacity=0.6
)
fig.update_layout(template='plotly_dark', yaxis_type='log')
fig.show()

### 2.4 ¿Coinciden los premios con la opinión del público?

In [ ]:
fig = px.scatter(
    pelis,
    x='averageRating', y='revenue_M',
    color='nominadas',
    hover_name='title',
    color_discrete_map={True: '#D4AF37', False: '#4A7FB5'},
    labels={
        'averageRating': 'Valoración IMDB',
        'revenue_M': 'Recaudación (mill $)',
        'nominadas': 'Nominada al Óscar'
    },
    title='Valoración del público vs. Recaudación',
    opacity=0.6
)
fig.update_layout(template='plotly_dark', yaxis_type='log')
fig.show()

In [ ]:
nominadas_r = pelis[pelis['nominadas'] == True]['averageRating'].dropna()
no_nominadas_r = pelis[pelis['nominadas'] == False]['averageRating'].dropna()

print(f"Valoración media nominadas:    {nominadas_r.mean():.2f}")
print(f"Valoración media no nominadas: {no_nominadas_r.mean():.2f}")

t, p = stats.ttest_ind(nominadas_r, no_nominadas_r)
print(f"\nt-test: t={t:.2f}, p={p:.4f}")
print("Diferencia estadísticamente significativa" if p < 0.05 else "No hay diferencia significativa")

Las nominadas tienen una valoración media notablemente superior, con una diferencia estadísticamente significativa (p < 0.05). Los Óscar sí tienden a premiar películas bien valoradas por el público, aunque no es una regla absoluta.

### 2.5 Géneros: proporción de nominaciones y volumen

In [ ]:
nominaciones_genero = (
    pelis[pelis['genre'] != 'TV Movie']
    .groupby('genre')
    .agg(total=('nominadas', 'count'), media=('nominadas', 'mean'))
    .assign(media=lambda x: (x['media'] * 100).round(1))
    .sort_values('media', ascending=False)
    .reset_index()
)

fig = make_subplots(rows=1, cols=2,
    subplot_titles=('% de películas nominadas por género', 'Volumen de películas por género'))

fig.add_trace(go.Bar(
    y=nominaciones_genero['genre'], x=nominaciones_genero['media'],
    orientation='h', marker_color='#D4AF37', name='% nominadas'
), row=1, col=1)

fig.add_trace(go.Bar(
    y=nominaciones_genero['genre'], x=nominaciones_genero['total'],
    orientation='h', marker_color='#4A7FB5', name='Total películas'
), row=1, col=2)

fig.update_layout(
    template='plotly_dark', height=500, showlegend=False,
    title='Nominaciones al Óscar por género'
)
fig.show()

**Observaciones:**
- **Horror** es el género más infra-nominado (~1.5%) pese a ser de los más producidos
- **Western** e **History** tienen tasas de nominación muy altas, pero con un volumen de películas muy reducido
- **Drama** domina tanto en volumen como en nominaciones absolutas

### 2.6 Rentabilidad (ROI) por género

In [ ]:
genero_rentabilidad = (
    pelis[pelis['genre'] != 'TV Movie']
    .groupby('genre')
    .agg(**{
        'Media ROI': ('ROI', 'mean'),
        'Mediana ROI': ('ROI', 'median'),
        'Media presupuesto (mill $)': ('budget_M', 'mean')
    })
    .round(1)
    .sort_values('Mediana ROI', ascending=False)
    .reset_index()
)

genero_rentabilidad

In [ ]:
fig = go.Figure()

fig.add_trace(go.Bar(
    y=genero_rentabilidad['genre'],
    x=genero_rentabilidad['Mediana ROI'],
    orientation='h',
    marker_color='#4A7FB5',
    name='Mediana ROI'
))

fig.add_vline(
    x=pelis['ROI'].median(), line_dash='dash', line_color='red',
    annotation_text='Mediana global', annotation_position='top right'
)

fig.update_layout(
    template='plotly_dark',
    title='Rentabilidad por género (mediana del ROI)',
    xaxis_title='ROI mediana',
    height=500
)
fig.show()

In [ ]:
# Distribución del ROI por género — la escala log revela que los outliers dominan la media
fig = px.box(
    pelis[pelis['genre'] != 'TV Movie'],
    y='genre', x='ROI',
    color_discrete_sequence=['#4A7FB5'],
    title='Distribución del ROI por género (escala logarítmica)',
    labels={'ROI': 'ROI', 'genre': 'Género'}
)
fig.update_layout(template='plotly_dark', xaxis_type='log')
fig.show()

**La mediana como métrica más robusta:**  
Usando la media, Horror aparece como el género más rentable — pero está distorsionado por pocos éxitos extraordinarios (*Paranormal Activity*, *The Blair Witch Project*). La mediana revela que la rentabilidad típica del género es mucho más moderada.

**Perfiles por género:**
- **Alto presupuesto, ROI moderado** (Action, Adventure, Animation, Sci-Fi): inversiones grandes con retornos predecibles
- **Bajo presupuesto, ROI variable** (Horror, Documentary): pueden ser muy rentables o perder dinero

### 2.7 ¿Las nominadas son más rentables?

In [ ]:
roi_nom = (
    pelis[pelis['genre'] != 'TV Movie']
    .groupby(['genre', 'nominadas'])['ROI']
    .mean()
    .reset_index()
)

fig = px.bar(
    roi_nom,
    y='genre', x='ROI', color='nominadas',
    barmode='group',
    color_discrete_map={True: '#D4AF37', False: '#4A7FB5'},
    labels={'ROI': 'ROI medio', 'genre': 'Género', 'nominadas': 'Nominada'},
    title='ROI medio: nominadas vs. no nominadas por género'
)
fig.update_layout(template='plotly_dark', height=500)
fig.show()

**Patrón general:** en la mayoría de géneros, las nominadas tienen un ROI similar o ligeramente superior.

**Excepciones notables:**
- **Documentary y Mystery:** las nominadas multiplican su ROI — aunque hay que leerlo con cautela: pocas películas en estas categorías reciben nominaciones y basta un éxito puntual para distorsionar la media
- **Horror:** sucede lo contrario. Los grandes éxitos de bajo coste del género (*Paranormal Activity*, *Get Out*) no suelen optar a premios

### 2.8 Las grandes ganadoras: ¿qué tienen en común?

Se comparan las películas con **3 o más Óscar** frente al resto.

In [ ]:
from wordcloud import WordCloud

frecuencias = dict(zip(
    pelis[pelis['awards'] >= 5]['title'],
    pelis[pelis['awards'] >= 5]['awards'].astype(int)
))

wc = WordCloud(
    background_color='#00002e', width=1200, height=500,
    colormap='YlOrBr', max_font_size=150, min_font_size=20
).generate_from_frequencies(frecuencias)

fig, ax = plt.subplots(figsize=(14, 6), facecolor='#00002e')
ax.imshow(wc, interpolation='bilinear')
ax.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
big_ones = pelis[pelis['awards'] >= 3].copy()
rest = pelis[pelis['awards'] < 3].copy()
rest = rest[rest['runtime'] > 1]  # Eliminar cortos y errores de duración

print(f"Grandes ganadoras (>=3 Oscar): {len(big_ones)} peliculas")
print(f"Resto: {len(rest):,} peliculas\n")

print("--- Grandes ganadoras ---")
print(big_ones[['budget', 'revenue', 'ROI', 'averageRating', 'runtime']].describe().round(1))
print("\n--- Resto ---")
print(rest[['budget', 'revenue', 'ROI', 'averageRating', 'runtime']].describe().round(1))

In [ ]:
big_ones['budget_M'] = big_ones['budget'] / 1e6
big_ones['revenue_M'] = big_ones['revenue'] / 1e6
rest['budget_M'] = rest['budget'] / 1e6
rest['revenue_M'] = rest['revenue'] / 1e6

metricas = ['budget_M', 'revenue_M', 'ROI', 'averageRating']
titulos = ['Presupuesto (Mill $)', 'Recaudacion (Mill $)', 'ROI', 'Valoracion IMDB']

fig = make_subplots(rows=2, cols=2, subplot_titles=titulos,
    vertical_spacing=0.15, horizontal_spacing=0.1)

colores = ['#4A90D9', '#D4AF37']

for i, (metrica, titulo) in enumerate(zip(metricas, titulos)):
    row, col = (i // 2) + 1, (i % 2) + 1
    valores = [rest[metrica].mean(), big_ones[metrica].mean()]
    fig.add_trace(go.Bar(
        x=['Resto', 'Grandes ganadoras'],
        y=valores,
        marker_color=colores,
        text=[f"{v:.1f}" for v in valores],
        textposition='auto',
        showlegend=False
    ), row=row, col=col)

fig.update_layout(
    template='plotly_dark',
    title='Grandes ganadoras (>=3 Oscar) vs. resto',
    height=600, width=900
)
fig.show()

**Diferencias clave:**
- Doblan el presupuesto medio (~70M $ vs ~35M $)
- Casi quintuplican la recaudación media
- ROI prácticamente el doble
- Valoración IMDB significativamente superior
- ~25 minutos más largas de media

> Una película con múltiples Óscar no es solo un éxito crítico y comercial: es también una apuesta de mayor riesgo que, cuando funciona, devuelve mucho más.

### 2.9 Distribución de géneros: grandes ganadoras vs. resto

In [ ]:
genre_big = big_ones['genre'].value_counts(normalize=True).mul(100).reset_index()
genre_big.columns = ['genre', 'pct']
genre_big['grupo'] = 'Grandes ganadoras'

genre_rest = rest['genre'].value_counts(normalize=True).mul(100).reset_index()
genre_rest.columns = ['genre', 'pct']
genre_rest['grupo'] = 'Resto'

genre_combined = pd.concat([genre_big, genre_rest])

fig = px.bar(
    genre_combined,
    y='genre', x='pct', color='grupo',
    barmode='group',
    color_discrete_map={'Grandes ganadoras': '#D4AF37', 'Resto': '#4A7FB5'},
    labels={'pct': '% del grupo', 'genre': 'Genero', 'grupo': ''},
    title='Distribucion de generos: grandes ganadoras vs. resto'
)
fig.update_layout(template='plotly_dark', height=550)
fig.show()

**Conclusiones sobre géneros y Óscar:**
- **Drama** domina entre las grandes ganadoras: ~50% vs ~23% en el resto. Una de cada dos películas con 3+ Óscar es un drama
- **Horror** casi desaparece entre los ganadores: es habitual en el resto (~6.5%) pero la Academia lo ignora sistemáticamente
- **Comedy** cae a la mitad: ~22% del resto vs ~9% entre grandes ganadoras

---
## 3. Conclusiones

| Dimensión | Hallazgo principal |
|---|---|
| **Presupuesto** | Correlación clara con recaudación (r=0.76), pero no con rentabilidad proporcional |
| **Géneros rentables** | Horror y Documentary destacan en ROI pero con alta varianza; Action y Sci-Fi dan retornos más predecibles |
| **Premios y público** | Las nominadas tienen valoración IMDB significativamente superior (p<0.05) |
| **Grandes ganadoras** | Doblan presupuesto, casi quintuplican recaudación, duran más y son mejor valoradas |
| **Géneros premiados** | Drama domina los Óscar; Horror y Comedy son sistemáticamente ignorados por la Academia |